# What is Artificial Intelligence? Hands-on notebook

**AI Agent Engineering Academy** · Track 01: AI and LLM Fundamentals · Module 1.1 · Beginner

This notebook is the interactive companion to the article
[What is Artificial Intelligence?](../../../docs/01-AI-FUNDAMENTALS/01-introduction-to-artificial-intelligence/01-what-is-artificial-intelligence.md).
You will build one small project, **an AI that sorts customer support tickets**, three ways:

| Part | Approach | Big idea |
|---|---|---|
| 1 | **Symbolic (rule-based) AI** | A human writes the knowledge as rules. |
| 2 | **Machine learning** | The computer learns the rules from labelled examples. |
| 3 | **An AI agent** | A model wrapped in a loop: *perceive → decide → act → learn*. |

Each code cell is short and has an explanation above it.
**Run the cells from top to bottom** with `Shift + Enter`.

> No API keys are needed. Everything runs locally on your computer.

## 0. Setup

### 0.1 Install the library

We need **scikit-learn**, a popular Python library for classic machine learning.
It automatically installs **NumPy** and **SciPy**, which it uses for maths.

The `%pip` command below is a Jupyter *magic command*: it installs the package into the same Python
environment that this notebook is running in. You only need to run it once.

In [ ]:
%pip install -q scikit-learn==1.9.1

### 0.2 Check the versions

AI libraries change often, so it is good practice to print the versions you are using.
This notebook was tested with **Python 3.14.5** and **scikit-learn 1.9.1**. Python 3.10 or newer is required.

In [2]:
import sys
import sklearn

print("Python      :", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)

Python      : 3.14.5
scikit-learn: 1.9.1


## 1. The problem: sorting support tickets

Imagine a software company that receives hundreds of support messages a day. Each one must go to the right team (**queue**):

| Queue | What belongs there |
|---|---|
| `billing` | payments, invoices, refunds, prices |
| `technical` | bugs, errors, crashes, slowness |
| `account` | login, passwords, profile, team access |

People describe the same problem in many different ways ("refund me", "charged twice", "money back"),
so we cannot write a perfect `if` statement for every message. **This is exactly the kind of problem AI is for.**

### 1.1 Shared building blocks

Before building any AI, we define two things that every approach will share:

1. **`Prediction`**: the answer an AI gives. It has a `label` (the queue), a `confidence` (0.0 to 1.0) and a `reason`.
2. **`TicketClassifier`**: a *contract* (interface). Anything with a `name` and a `predict(text)` method counts as a classifier.

Why bother? Because later our agent can work with **any** classifier (rules, machine learning or, later in the course, a large language model)
without changing a single line of the agent's code.

**Code notes**

- `@dataclass(frozen=True)` writes the boring `__init__` and `__repr__` methods for us. `frozen=True` means a prediction cannot be changed after it is created.
- `Protocol` describes the *shape* a class must have. Classes do not need to inherit from it; they only need matching methods.

In [3]:
from dataclasses import dataclass
from typing import Protocol


@dataclass(frozen=True)
class Prediction:
    # The result of classifying one ticket.
    label: str          # e.g. "billing"
    confidence: float   # 0.0 (no idea) .. 1.0 (very sure)
    reason: str         # short human-readable explanation


class TicketClassifier(Protocol):
    # Anything with a `name` and a `predict(text) -> Prediction` method.
    name: str

    def predict(self, text: str) -> Prediction: ...


# Quick check: create a Prediction by hand and print it.
example = Prediction(label="billing", confidence=0.9, reason="just an example")
print(example)

Prediction(label='billing', confidence=0.9, reason='just an example')


### 1.2 The data

Machine learning needs **labelled data**: examples where we already know the right answer.
Each example below is a pair `(text, label)`.

We make **two separate lists**:

- `TRAINING_DATA` (30 tickets): the examples the model is allowed to learn from.
- `TEST_DATA` (9 tickets): new tickets kept hidden during training. We use them at the end to check whether the AI **generalises**, meaning it works on wording it has never seen.

> Real systems use thousands or millions of examples. We keep it tiny so you can read every line.

In [4]:
LABELS = ("billing", "technical", "account")

TRAINING_DATA = [
    # billing
    ("I was charged twice for my subscription this month", "billing"),
    ("Please send me the invoice for last quarter", "billing"),
    ("How do I get a refund for the annual plan", "billing"),
    ("My credit card payment was declined", "billing"),
    ("Why did my monthly price go up", "billing"),
    ("I want to change my payment method to PayPal", "billing"),
    ("There is an unknown charge on my bank statement", "billing"),
    ("Can I get a discount if I pay yearly", "billing"),
    ("The receipt shows the wrong company name", "billing"),
    ("Cancel my plan and stop billing me", "billing"),
    # technical
    ("The app crashes every time I open the dashboard", "technical"),
    ("I get error 500 when uploading a file", "technical"),
    ("The page is very slow to load since the update", "technical"),
    ("Export to CSV produces an empty file", "technical"),
    ("Notifications stopped working on Android", "technical"),
    ("The search bar returns no results for anything", "technical"),
    ("Images are not displaying in the report", "technical"),
    ("The API returns a timeout after 30 seconds", "technical"),
    ("Sync between my laptop and phone is broken", "technical"),
    ("The button does nothing when I click save", "technical"),
    # account
    ("I forgot my password and cannot log in", "account"),
    ("How do I change the email address on my profile", "account"),
    ("Please delete my account and personal data", "account"),
    ("My account is locked after too many attempts", "account"),
    ("I want to add a new team member to our workspace", "account"),
    ("Two-factor authentication code is not arriving", "account"),
    ("How can I change my username", "account"),
    ("Someone else may have accessed my account", "account"),
    ("I need admin permissions for my colleague", "account"),
    ("Sign in with Google is not linking to my profile", "account"),
]

TEST_DATA = [
    ("I paid but my order still shows unpaid", "billing"),
    ("You billed my card after I cancelled", "billing"),
    ("Where can I download my receipts", "billing"),
    ("The application freezes when I scroll the table", "technical"),
    ("Uploading a photo gives me a strange error", "technical"),
    ("Charts are blank after the latest release", "technical"),
    ("I cannot reset my password, the link has expired", "account"),
    ("Please remove a former employee from our team", "account"),
    ("My login keeps failing even with the right password", "account"),
]

from collections import Counter

print("Training examples:", len(TRAINING_DATA), dict(Counter(label for _, label in TRAINING_DATA)))
print("Test examples    :", len(TEST_DATA), dict(Counter(label for _, label in TEST_DATA)))

Training examples: 30 {'billing': 10, 'technical': 10, 'account': 10}
Test examples    : 9 {'billing': 3, 'technical': 3, 'account': 3}


Notice the data is **balanced**: 10 training examples per queue. If one queue had 100 examples and another had 2,
the model would learn to favour the big queue.

## 2. Approach 1: Symbolic (rule-based) AI

This is how AI was mostly built from the 1950s to the 1980s. A human expert writes the knowledge down as rules:

> *If the ticket contains "refund" or "invoice", it is probably about billing.*

**How our version works**

1. Each queue has a list of keywords (the **knowledge base**).
2. We count how many keywords from each queue appear in the ticket (the **inference engine**).
3. The queue with the most matches wins.
4. **Confidence** = the winning queue's matches ÷ all matches. If *nothing* matches, confidence is `0.0` and we fall back to `technical`.

**Code notes**

- `re.search(rf"\b{re.escape(keyword)}\b", text)`: `\b` means *word boundary*, so the keyword `card` does **not** match inside the word `discarded`. `re.escape` makes characters such as `-` in `two-factor` literal.
- Empty text raises a `ValueError`. It is better to **fail fast** than to guess.

In [5]:
import re

DEFAULT_RULES = {
    "billing": (
        "charge", "charged", "invoice", "refund", "payment", "price",
        "billing", "bill", "discount", "receipt", "plan", "card",
    ),
    "technical": (
        "crash", "crashes", "error", "slow", "bug", "broken", "timeout",
        "not working", "stopped working", "empty", "api", "load",
    ),
    "account": (
        "password", "log in", "login", "sign in", "account", "username",
        "email", "profile", "permissions", "locked", "two-factor", "team",
    ),
}

FALLBACK_LABEL = "technical"


class RuleBasedClassifier:
    # Classify tickets by counting keyword matches per queue.

    name = "rule-based"

    def __init__(self, rules=None):
        self.rules = rules if rules is not None else DEFAULT_RULES

    @staticmethod
    def _matches(keyword: str, text: str) -> bool:
        return re.search(rf"\b{re.escape(keyword)}\b", text) is not None

    def predict(self, text: str) -> Prediction:
        if not text or not text.strip():
            raise ValueError("Ticket text must not be empty.")

        normalized = text.lower()
        # e.g. {"billing": ["refund"], "technical": [], "account": []}
        hits = {
            label: [kw for kw in keywords if self._matches(kw, normalized)]
            for label, keywords in self.rules.items()
        }
        total_hits = sum(len(found) for found in hits.values())

        if total_hits == 0:
            return Prediction(FALLBACK_LABEL, 0.0, "no rule matched; used fallback queue")

        best_label = max(hits, key=lambda label: len(hits[label]))
        confidence = len(hits[best_label]) / total_hits
        return Prediction(best_label, round(confidence, 2), f"matched keywords {hits[best_label]}")


rules = RuleBasedClassifier()
print("Classifier ready:", rules.name)

Classifier ready: rule-based


### 2.1 Try the rules

Run the cell below. Then **change the tickets** and run it again. Can you find a ticket that fools the rules?

In [6]:
for ticket in [
    "Please refund my last invoice",       # clear billing keywords
    "The app crashes with an error",       # clear technical keywords
    "Refund me, the app crashed",          # two topics at once
    "Where can I download my receipts",    # 'receipts' is not in the list, only 'receipt'
]:
    print(f"{ticket!r:40} -> {rules.predict(ticket)}")

'Please refund my last invoice'          -> Prediction(label='billing', confidence=1.0, reason="matched keywords ['invoice', 'refund']")
'The app crashes with an error'          -> Prediction(label='technical', confidence=1.0, reason="matched keywords ['crashes', 'error']")
'Refund me, the app crashed'             -> Prediction(label='billing', confidence=1.0, reason="matched keywords ['refund']")
'Where can I download my receipts'       -> Prediction(label='technical', confidence=0.0, reason='no rule matched; used fallback queue')


**What to notice**

- Clear tickets get confidence `1.0` and an explanation (`matched keywords [...]`). Rules are easy to **explain**.
- Look closely at `"Refund me, the app crashed"`. It is about two topics, yet the rules say `billing` with confidence `1.0`.
  Why? The rule list contains `crash` and `crashes`, but **not** `crashed`, so the technical half of the ticket is invisible to the rules.
  The rules are not just wrong here, they are *confidently* wrong.
- `"receipts"` matched **nothing**, because the rule list only has `receipt`. Rules are **brittle**: they only know exactly what a human typed in.
  Fixing this means a person must edit the rules again, forever. This is called the *knowledge acquisition bottleneck*.

## 3. Approach 2: Machine learning

Instead of writing rules, we show the computer the 30 labelled examples and let it **learn** which words point to which queue.

```
Traditional programming:  rules + data    --> answers
Machine learning:         data  + answers --> rules (a "model")
```

### 3.1 Step one: turning text into numbers (features)

Models only understand numbers. **TF-IDF** (*term frequency–inverse document frequency*) turns each ticket into a list of numbers:

- A word gets a **high** weight if it appears in this ticket but is **rare** across all tickets (for example `refund`).
- A word gets a **low** weight if it appears almost everywhere (for example `the`).

Let's look at it before building the full model. The cell below learns a vocabulary from the training texts
and shows the non-zero weights for one ticket.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

demo_vectorizer = TfidfVectorizer(lowercase=True)
demo_vectorizer.fit([text for text, _ in TRAINING_DATA])   # learn the vocabulary

print("Vocabulary size:", len(demo_vectorizer.vocabulary_), "words")

vector = demo_vectorizer.transform(["Please refund my invoice"])
words = demo_vectorizer.get_feature_names_out()
for index, weight in zip(vector.indices, vector.data):
    print(f"  {words[index]:<10} {weight:.2f}")

Vocabulary size: 150 words
  invoice    0.58
  my         0.26
  please     0.52
  refund     0.58


Only words that appeared in the training data have a column. `please` and `my` are common, so they get lower weights;
`refund` and `invoice` are rarer and more informative, so they get higher weights. Every other word in the vocabulary is `0` for this ticket.

### 3.2 Step two: the learning algorithm

We use **logistic regression**, a simple and fast classification algorithm. During training it learns one **weight** per *(word, queue)* pair:

- a positive weight means "this word suggests this queue";
- a negative weight means "this word suggests *not* this queue".

To predict, it adds up the weights of the words in the ticket and converts the totals into **probabilities** that sum to 1.
The highest probability becomes the label, and its value becomes the **confidence**.

**Code notes**

- `Pipeline` chains the two steps, so training and prediction always use exactly the same text-to-numbers conversion.
- `ngram_range=(1, 2)` uses single words *and* pairs of words, so `"log in"` becomes its own feature.
- `C=10.0` is a **hyperparameter** (a setting we choose, not something learned). It controls *regularisation*, a penalty that stops the model from overfitting.
  The default `C=1.0` is too strict for only 30 examples and makes every prediction look unsure.
- `learn()` adds one corrected example and retrains. This is the simplest possible way to learn from feedback.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


class MLClassifier:
    # A text classifier that learns from labelled examples.

    name = "machine-learning"

    def __init__(self):
        self.pipeline = Pipeline(steps=[
            ("vectorizer", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
            ("model", LogisticRegression(C=10.0, max_iter=1000)),
        ])
        self._examples = []
        self.is_trained = False

    def train(self, examples):
        if len({label for _, label in examples}) < 2:
            raise ValueError("Training needs examples from at least two labels.")
        self._examples = list(examples)
        texts = [text for text, _ in self._examples]
        targets = [label for _, label in self._examples]
        self.pipeline.fit(texts, targets)        # <-- the actual learning happens here
        self.is_trained = True

    def learn(self, text, correct_label):
        self.train([*self._examples, (text, correct_label)])

    def predict(self, text: str) -> Prediction:
        if not self.is_trained:
            raise RuntimeError("Call train() before predict().")
        if not text or not text.strip():
            raise ValueError("Ticket text must not be empty.")

        probabilities = self.pipeline.predict_proba([text])[0]   # one row, one column per queue
        classes = self.pipeline.classes_                         # queue names, in column order
        best = int(probabilities.argmax())
        return Prediction(str(classes[best]), round(float(probabilities[best]), 2),
                          "learned from labelled examples")


model = MLClassifier()
model.train(TRAINING_DATA)
print("Trained on", len(TRAINING_DATA), "examples. Classes:", list(model.pipeline.classes_))

Trained on 30 examples. Classes: [np.str_('account'), np.str_('billing'), np.str_('technical')]


### 3.3 Try the trained model

Same tickets as before. Compare the answers with the rule-based ones.

In [9]:
for ticket in [
    "Please refund my last invoice",
    "The app crashes with an error",
    "Refund me, the app crashed",
    "I cannot log in after changing my phone",
]:
    print(f"{ticket!r:42} -> {model.predict(ticket)}")

'Please refund my last invoice'            -> Prediction(label='billing', confidence=0.61, reason='learned from labelled examples')
'The app crashes with an error'            -> Prediction(label='technical', confidence=0.69, reason='learned from labelled examples')
'Refund me, the app crashed'               -> Prediction(label='billing', confidence=0.48, reason='learned from labelled examples')
'I cannot log in after changing my phone'  -> Prediction(label='account', confidence=0.65, reason='learned from labelled examples')


### 3.4 Look inside the model

A big advantage of simple models is that we can **inspect what they learned**.
The cell below prints the words with the largest positive weight for each queue.
If these words make sense, the model has learned something sensible.

In [10]:
import numpy as np

vectorizer = model.pipeline.named_steps["vectorizer"]
classifier = model.pipeline.named_steps["model"]
feature_names = vectorizer.get_feature_names_out()

for row, label in enumerate(classifier.classes_):
    top = np.argsort(classifier.coef_[row])[-6:][::-1]     # indices of the 6 biggest weights
    print(f"{label:<10}", ", ".join(feature_names[i] for i in top))

account    account, my account, my, my profile, profile, is not
billing    payment, plan, me, was, get, for
technical  the, file, when, returns, working, working on


**What to notice:** some top words are sensible (`account`, `profile`, `payment`, `plan`), but others are noise:
`the` and `when` for technical, or `me`, `was` and `for` for billing. They appear there only because they happened to occur
more often in one queue's 10 examples. With so little data, the model picks up **accidental patterns** as well as real ones.
More (and more varied) training data is the main cure. You will see the same problem in a different form when we study bias later in the course.

### 3.5 Which words does the model actually "see"?

The model can only use words that were in its training data. Unknown words are silently ignored.
This helper shows which words of a ticket the model knows. (The default tokenizer also skips one-letter words such as "I".)

In [11]:
def known_words(text: str) -> list[str]:
    analyzer = vectorizer.build_analyzer()          # the same tokenizer the model uses
    return [w for w in analyzer(text) if w in vectorizer.vocabulary_]

for ticket in ["I paid but my order still shows unpaid", "Where can I download my receipts"]:
    print(f"{ticket!r:42} known words: {known_words(ticket)}")

'I paid but my order still shows unpaid'   known words: ['my', 'shows']
'Where can I download my receipts'         known words: ['can', 'my']


**Important lesson:** for the first ticket the model only knows `my` and `shows`. The word `shows` appeared once in training,
in a *billing* ticket ("The receipt **shows** the wrong company name"). So if the model says "billing", it is right **for a weak reason**.
On tiny datasets that happens a lot, which is why we measure on many test examples instead of trusting a few lucky ones.

## 4. Evaluation: which approach is better?

"It seems to work" is not evidence. We **measure** both approaches on the 9 hidden test tickets.

**Accuracy** = number of correct predictions ÷ number of predictions.

In [12]:
def accuracy(classifier, examples) -> float:
    if not examples:
        raise ValueError("Cannot compute accuracy on an empty dataset.")
    correct = sum(1 for text, expected in examples if classifier.predict(text).label == expected)
    return correct / len(examples)


for clf in (rules, model):
    print(f"{clf.name:<18} accuracy: {accuracy(clf, TEST_DATA):.0%}  ({len(TEST_DATA)} test tickets)")

rule-based         accuracy: 78%  (9 test tickets)
machine-learning   accuracy: 89%  (9 test tickets)


Now let's see every test ticket side by side. ✅ means correct, ❌ means wrong.

In [13]:
print(f"{'ticket':<52} {'expected':<10} {'rules':<12} {'ml':<12}")
print("-" * 88)
for text, expected in TEST_DATA:
    r = rules.predict(text).label
    m = model.predict(text).label
    mark = lambda label: ("✅ " if label == expected else "❌ ") + label
    print(f"{text:<52} {expected:<10} {mark(r):<12} {mark(m):<12}")

ticket                                               expected   rules        ml          
----------------------------------------------------------------------------------------
I paid but my order still shows unpaid               billing    ❌ technical  ✅ billing   
You billed my card after I cancelled                 billing    ✅ billing    ✅ billing   
Where can I download my receipts                     billing    ❌ technical  ❌ account   
The application freezes when I scroll the table      technical  ✅ technical  ✅ technical 
Uploading a photo gives me a strange error           technical  ✅ technical  ✅ technical 
Charts are blank after the latest release            technical  ✅ technical  ✅ technical 
I cannot reset my password, the link has expired     account    ✅ account    ✅ account   
Please remove a former employee from our team        account    ✅ account    ✅ account   
My login keeps failing even with the right password  account    ✅ account    ✅ account   


**What to notice**

- Machine learning should score higher than the rules, because it handles wording the rules never listed.
- Both get `"Where can I download my receipts"` wrong. The ML model has never seen `receipts` either, so it guesses from `can` and `my`.
  **Learning is not magic: a model cannot use information it has never seen.**
  Later in the course, *embeddings* and *large language models* solve this much better, because they understand that "receipts" and "receipt" mean nearly the same thing.

## 5. Approach 3: An AI agent

A model only **predicts**. An **agent** uses predictions to **act** toward a goal. Every agent in this academy follows the same loop:

```
        ┌────────────────────────────────────────────┐
        ▼                                            │
   PERCEIVE ──► DECIDE ──► ACT ──► (remember) ──► LEARN from feedback
   clean the    is the      put the ticket         a human corrects
   input        model sure? in a queue             a mistake
```

Our `TriageAgent` adds four things on top of a classifier:

| Part | Method | What it does |
|---|---|---|
| Perception | `perceive()` | Cleans up the raw text and rejects empty tickets. |
| Decision policy | `decide()` | Acts alone only if confidence ≥ `confidence_threshold`; otherwise sends the ticket to a **human**. |
| Action | `act()` | Changes the world: appends the ticket to a queue. |
| Memory + learning | `handle()`, `receive_feedback()` | Remembers every step; lets a human correct it. |

**Code notes**

- `field(default_factory=...)` gives each agent its *own* fresh queues and memory (dataclasses must not share one mutable default).
- `defaultdict(list)` creates an empty list automatically the first time a new queue name is used.
- `receive_feedback` checks whether the classifier *can* learn (`getattr(..., "learn", None)`). The rule-based classifier cannot, so it returns `False`.

In [14]:
from collections import defaultdict
from dataclasses import field

HUMAN_REVIEW_QUEUE = "human-review"


@dataclass
class AgentStep:
    # One record in the agent's memory.
    ticket: str
    prediction: Prediction
    action: str


@dataclass
class TriageAgent:
    classifier: TicketClassifier
    confidence_threshold: float = 0.5
    queues: dict = field(default_factory=lambda: defaultdict(list))
    memory: list = field(default_factory=list)

    def perceive(self, raw_ticket: str) -> str:
        text = " ".join(raw_ticket.split())        # collapse extra spaces
        if not text:
            raise ValueError("Received an empty ticket.")
        return text

    def decide(self, ticket: str):
        prediction = self.classifier.predict(ticket)
        if prediction.confidence >= self.confidence_threshold:
            return prediction, prediction.label     # confident -> act alone
        return prediction, HUMAN_REVIEW_QUEUE       # unsure -> ask a human

    def act(self, ticket: str, queue: str) -> str:
        self.queues[queue].append(ticket)
        return f"routed to '{queue}'"

    def handle(self, raw_ticket: str) -> AgentStep:
        ticket = self.perceive(raw_ticket)
        prediction, queue = self.decide(ticket)
        action = self.act(ticket, queue)
        step = AgentStep(ticket, prediction, action)
        self.memory.append(step)
        return step

    def receive_feedback(self, ticket: str, correct_label: str) -> bool:
        learn = getattr(self.classifier, "learn", None)
        if callable(learn):
            learn(ticket, correct_label)
            return True
        return False


print("TriageAgent defined.")

TriageAgent defined.


### 5.1 Run the agent on new tickets

The agent uses the trained machine-learning model as its "brain" and a confidence threshold of `0.5`.

In [15]:
agent = TriageAgent(classifier=model, confidence_threshold=0.5)

incoming = [
    "I was billed twice and need my money back",
    "The app shows error 403 when I open settings",
    "I cannot log in after changing my phone",
    "Hello, quick question about your product",
]

for ticket in incoming:
    step = agent.handle(ticket)
    print(f"Ticket : {step.ticket}")
    print(f"Model  : {step.prediction.label} (confidence {step.prediction.confidence:.2f})")
    print(f"Action : {step.action}\n")

print("Queue sizes:", {queue: len(items) for queue, items in agent.queues.items()})

Ticket : I was billed twice and need my money back
Model  : billing (confidence 0.46)
Action : routed to 'human-review'

Ticket : The app shows error 403 when I open settings
Model  : technical (confidence 0.71)
Action : routed to 'technical'

Ticket : I cannot log in after changing my phone
Model  : account (confidence 0.65)
Action : routed to 'account'

Ticket : Hello, quick question about your product
Model  : technical (confidence 0.35)
Action : routed to 'human-review'

Queue sizes: {'human-review': 2, 'technical': 1, 'account': 1}


**What to notice**

- Confident tickets are routed automatically.
- `"I was billed twice..."` may be predicted *correctly* but with low confidence, because the model has seen "billing" but never "billed". The agent plays safe and escalates.
- `"Hello, quick question..."` does not belong to any queue. A plain classifier would force it into one anyway; the agent sends it to a human instead.

### 5.2 The agent's memory

Everything the agent did is stored, which is useful for **auditing** and **debugging**: you can always see *why* it made a decision.

In [16]:
for number, step in enumerate(agent.memory, start=1):
    print(f"{number}. {step.action:<26} label={step.prediction.label:<10} "
          f"conf={step.prediction.confidence:.2f}  ticket={step.ticket!r}")

1. routed to 'human-review'   label=billing    conf=0.46  ticket='I was billed twice and need my money back'
2. routed to 'technical'      label=technical  conf=0.71  ticket='The app shows error 403 when I open settings'
3. routed to 'account'        label=account    conf=0.65  ticket='I cannot log in after changing my phone'
4. routed to 'human-review'   label=technical  conf=0.35  ticket='Hello, quick question about your product'


### 5.3 Learning from human feedback

A human reviews the escalated billing ticket and confirms that it really is `billing`.
The agent passes this correction to the model, which retrains. Watch the confidence change.

In [17]:
ticket = "I was billed twice and need my money back"

print("Before feedback:", model.predict(ticket))
learned = agent.receive_feedback(ticket, correct_label="billing")
print("Classifier learned from the correction:", learned)
print("After feedback :", model.predict(ticket))

Before feedback: Prediction(label='billing', confidence=0.46, reason='learned from labelled examples')
Classifier learned from the correction: True
After feedback : Prediction(label='billing', confidence=0.85, reason='learned from labelled examples')


The confidence goes up because the model now knows the words `billed`, `money` and `back`.
A rule-based agent **cannot** do this on its own. Let's prove it:

In [18]:
rule_agent = TriageAgent(classifier=RuleBasedClassifier())
print("Rule-based agent learned:", rule_agent.receive_feedback(ticket, "billing"))

Rule-based agent learned: False


> ⚠️ **Production note:** retraining instantly on every correction is fine for a demo, but risky in real systems.
> A wrong or malicious correction would change behaviour immediately (*data poisoning*). Real systems collect corrections,
> review them, and retrain in batches only if a test set shows the new model is at least as good.

## 6. Experiment: the confidence threshold

The threshold is a **policy** decision, not part of the model. It trades **automation** (fewer tickets for humans)
against **safety** (fewer automatic mistakes). Run the cell and see how many tickets reach a human at each setting.

We train a *fresh* model here so the feedback from section 5.3 does not affect the experiment.

In [19]:
fresh_model = MLClassifier()
fresh_model.train(TRAINING_DATA)
all_tickets = [text for text, _ in TEST_DATA] + incoming

print(f"{'threshold':<10} {'handled automatically':<24} {'sent to a human'}")
for threshold in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    test_agent = TriageAgent(classifier=fresh_model, confidence_threshold=threshold)
    for text in all_tickets:
        test_agent.handle(text)
    to_human = len(test_agent.queues[HUMAN_REVIEW_QUEUE])
    print(f"{threshold:<10} {len(all_tickets) - to_human:<24} {to_human}")

threshold  handled automatically    sent to a human
0.3        13                       0
0.4        11                       2
0.5        8                        5
0.6        4                        9
0.7        2                        11
0.8        0                        13


Higher threshold → more tickets go to humans (safer, but more work).
Lower threshold → more automation (cheaper, but more mistakes slip through).
Real teams pick the threshold by measuring both on held-out data and choosing what the business can accept.

## 7. Testing AI code

AI components are still software, so we test them. Two useful techniques:

1. **Test the deterministic parts exactly**: input validation, routing rules, error handling.
2. **Use a fake classifier** (a *test double*) to test the agent's policy without depending on what the real model predicts.
   You will use the same trick later to test LLM agents without paying for API calls.

The cell below defines small test functions with `assert` and runs them all. If nothing fails, every check passed.
(The companion folder also has the same tests for **pytest** in `tests/test_triage.py`.)

In [20]:
class FixedClassifier:
    # A fake classifier that always returns the same answer.
    name = "fixed"

    def __init__(self, label, confidence):
        self._prediction = Prediction(label, confidence, "fixed for testing")

    def predict(self, text):
        return self._prediction


def test_rules_match_keywords():
    p = RuleBasedClassifier().predict("Please refund my last invoice")
    assert p.label == "billing" and p.confidence == 1.0

def test_rules_use_word_boundaries():
    assert RuleBasedClassifier().predict("My draft was discarded").confidence == 0.0

def test_ml_requires_training():
    try:
        MLClassifier().predict("anything")
    except RuntimeError:
        return
    raise AssertionError("expected RuntimeError")

def test_ml_returns_valid_prediction():
    p = fresh_model.predict("I was charged twice")
    assert p.label in LABELS and 0.0 <= p.confidence <= 1.0

def test_agent_acts_when_confident():
    a = TriageAgent(FixedClassifier("billing", 0.9))
    step = a.handle("  refund   please ")
    assert step.ticket == "refund please"
    assert a.queues["billing"] == ["refund please"]

def test_agent_escalates_when_unsure():
    a = TriageAgent(FixedClassifier("billing", 0.2))
    a.handle("unclear message")
    assert a.queues[HUMAN_REVIEW_QUEUE] == ["unclear message"]


tests = [test_rules_match_keywords, test_rules_use_word_boundaries, test_ml_requires_training,
         test_ml_returns_valid_prediction, test_agent_acts_when_confident, test_agent_escalates_when_unsure]
for test in tests:
    test()
    print("PASSED", test.__name__)

PASSED test_rules_match_keywords
PASSED test_rules_use_word_boundaries
PASSED test_ml_requires_training
PASSED test_ml_returns_valid_prediction
PASSED test_agent_acts_when_confident
PASSED test_agent_escalates_when_unsure


## 8. Try it yourself

Type any ticket into the cell below and run it. The agent will show what it predicts and what it would do.

In [21]:
my_ticket = "My invoice shows the wrong amount and the page keeps crashing"   # <-- edit me

playground = TriageAgent(classifier=fresh_model, confidence_threshold=0.5)
step = playground.handle(my_ticket)
print("Rules :", rules.predict(my_ticket))
print("ML    :", step.prediction)
print("Agent :", step.action)
print("Known words:", known_words(my_ticket))

Rules : Prediction(label='billing', confidence=1.0, reason="matched keywords ['invoice']")
ML    : Prediction(label='billing', confidence=0.56, reason='learned from labelled examples')
Agent : routed to 'billing'
Known words: ['my', 'invoice', 'shows', 'the', 'wrong', 'and', 'the', 'page', 'shows the', 'the wrong', 'the page']


## 9. Exercises

**Beginner**

1. Add keywords to `DEFAULT_RULES` so the rules handle `"Where can I download my receipts"`. Re-run sections 2 and 4.
2. Write five of your own tickets in section 8. Include one that mixes two topics.
3. Change `confidence_threshold` in section 5.1 to `0.3` and `0.8`. What changes?

**Intermediate**

4. Add a fourth queue, `sales` (pricing questions, demos, upgrades), with at least 8 training and 3 test examples.
5. Use `from sklearn.metrics import classification_report` to print precision and recall per queue for both classifiers.
6. In section 3.2, try `C=1.0` and `C=100.0`. How do accuracy and confidence change? Why?

**Advanced**

7. Build a `HybridClassifier` that trusts the rules when their confidence is `1.0` and uses the ML model otherwise. Compare all three in section 4.
8. Make feedback *safe*: store corrections in a pending list and only retrain if accuracy on `TEST_DATA` does not drop.

## 10. Summary

- **AI** is software that works out good outputs (predictions, decisions, actions) for inputs nobody wrote an exact rule for.
- **Rule-based AI** is explainable and predictable, but brittle and cannot learn.
- **Machine learning** learns from labelled data via **features → model → probabilities**, but cannot use words it has never seen.
- **Evaluation** on hidden test data tells you which approach is really better.
- An **agent** wraps a model with *perceive → decide → act → learn*, and knows when to ask a human.

**Next lesson:** [History of Artificial Intelligence](../../../docs/01-AI-FUNDAMENTALS/01-introduction-to-artificial-intelligence/02-history-of-artificial-intelligence.md)